# 从对偶到 CCG 的 max-min oracle

这个 notebook 辅助说明 `dual_cgg.md` 的完整过程：

1. 用资源约束线性组合引出对偶问题。
2. 解释影子价格和强对偶。
3. 用一个补救运行问题展示如何把内层最小化替换成对偶最大化。
4. 用电热补救例子解释 $q,W,h(d),T$ 和对偶变量。
5. 说明 CCG 的 max-min oracle 如何变成单层最大化问题。

## 1. 原问题：资源有限时最大化利润

$$
\begin{aligned}
\max_{x_1,x_2}
\quad & 3x_1+2x_2 \\
\text{s.t.}
\quad & x_1+x_2\le4 \\
& 2x_1+x_2\le5 \\
& x_1,x_2\ge0
\end{aligned}
$$

这里 $x_1,x_2$ 是两个产品的生产数量，两条不等式是两类资源约束。

In [ ]:
import gurobipy as gp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from gurobipy import GRB

env = gp.Env(empty=True)
env.setParam("OutputFlag", 0)
env.start()

p = gp.Model("resource_primal", env=env)
x1 = p.addVar(lb=0.0, name="x1")  # 产品1产量
x2 = p.addVar(lb=0.0, name="x2")  # 产品2产量
r1 = p.addConstr(x1 + x2 <= 4, name="resource_1")  # 资源1上限
r2 = p.addConstr(2 * x1 + x2 <= 5, name="resource_2")  # 资源2上限
p.setObjective(3 * x1 + 2 * x2, GRB.MAXIMIZE)  # 最大化利润
p.optimize()

pd.DataFrame([
    ("x1", x1.X),
    ("x2", x2.X),
    ("primal objective", p.ObjVal),
    ("resource 1 slack", r1.Slack),
    ("resource 2 slack", r2.Slack),
], columns=["item", "value"])

## 2. 从“上界证明”引出对偶

把第一条资源约束乘以 $y_1\ge0$，第二条资源约束乘以 $y_2\ge0$，再相加：

$$
(y_1+2y_2)x_1+(y_1+y_2)x_2\le4y_1+5y_2
$$

如果：

$$
y_1+2y_2\ge3
$$

$$
y_1+y_2\ge2
$$

那么：

$$
3x_1+2x_2\le4y_1+5y_2
$$

所以 $4y_1+5y_2$ 是原问题目标值的一个上界。找最紧的上界，就得到对偶问题。

## 3. 对偶问题

$$
\begin{aligned}
\min_{y_1,y_2}
\quad & 4y_1+5y_2 \\
\text{s.t.}
\quad & y_1+2y_2\ge3 \\
& y_1+y_2\ge2 \\
& y_1,y_2\ge0
\end{aligned}
$$

$y_1,y_2$ 是资源 1、资源 2 的影子价格。

In [ ]:
d = gp.Model("resource_dual", env=env)
y1 = d.addVar(lb=0.0, name="y1")  # 资源1影子价格
y2 = d.addVar(lb=0.0, name="y2")  # 资源2影子价格
pc1 = d.addConstr(y1 + 2 * y2 >= 3, name="product_1_profit_cover")  # 产品1利润覆盖
pc2 = d.addConstr(y1 + y2 >= 2, name="product_2_profit_cover")  # 产品2利润覆盖
d.setObjective(4 * y1 + 5 * y2, GRB.MINIMIZE)  # 最小化资源总估值
d.optimize()

pd.DataFrame([
    ("y1", y1.X),
    ("y2", y2.X),
    ("dual objective", d.ObjVal),
    ("product 1 dual slack", pc1.Slack),
    ("product 2 dual slack", pc2.Slack),
    ("duality gap", d.ObjVal - p.ObjVal),
], columns=["item", "value"])

## 4. 互补松弛和影子价格

互补松弛：

$$
y_i(b_i-A_i x)=0
$$

$$
x_j((A^\top y)_j-c_j)=0
$$

含义是：没用完的资源影子价格为 $0$；影子价格为正的资源一定用满。真正生产的产品，其资源影子价值刚好等于利润。

In [ ]:
pd.DataFrame([
    ("y1 * resource_1_slack", y1.X * r1.Slack),
    ("y2 * resource_2_slack", y2.X * r2.Slack),
    ("x1 * product_1_dual_slack", x1.X * pc1.Slack),
    ("x2 * product_2_dual_slack", x2.X * pc2.Slack),
], columns=["complementary slackness item", "value"])

## 5. CCG 里的内层运行问题从哪里来

固定第 $k$ 轮主问题得到的投资方案 $x^k$，不确定需求为 $d$，补救运行变量为 $y$。

最简单运行约束是：

$$
x^k+y\ge d
$$

也就是：

$$
y\ge d-x^k
$$

这就是抽象形式 $Wy\ge h(d)-Tx^k$ 的一维版本，其中 $W=1$，$h(d)=d$，$T=1$。

## 6. 内层原问题：运行补救成本

$$
Q(x^k,d)=\min_y 5y
$$

$$
\text{s.t.}\quad y\ge d-x^k
$$

$$
y\ge0
$$

这个问题的物理含义是：如果需求超过已建容量，就用补救量 $y$ 补上，单位补救成本为 $5$。

In [ ]:
x_fixed = 3.0
d_fixed = 10.0

rp = gp.Model("recourse_primal", env=env)
yy = rp.addVar(lb=0.0, name="y")  # 补救运行量
rp.addConstr(yy >= d_fixed - x_fixed, name="rescue_need")  # 补救量覆盖需求缺口
rp.setObjective(5 * yy, GRB.MINIMIZE)  # 最小化补救成本
rp.optimize()

pd.DataFrame([
    ("x_fixed", x_fixed),
    ("d_fixed", d_fixed),
    ("y", yy.X),
    ("Q primal", rp.ObjVal),
], columns=["item", "value"])

## 7. 内层对偶问题

原问题：

$$
\min_y\{5y\mid y\ge d-x^k,\ y\ge0\}
$$

给约束 $y\ge d-x^k$ 配对偶变量 $\pi\ge0$。因为原变量 $y\ge0$，对偶约束是 $\pi\le5$。

对偶问题为：

$$
\max_\pi \pi(d-x^k)
$$

$$
\text{s.t.}\quad 0\le\pi\le5
$$

In [ ]:
rd = gp.Model("recourse_dual", env=env)
pi = rd.addVar(lb=0.0, ub=5.0, name="pi")  # 需求缺口约束的影子价格
rd.setObjective(pi * (d_fixed - x_fixed), GRB.MAXIMIZE)  # 最大化对偶目标
rd.optimize()

pd.DataFrame([
    ("pi", pi.X),
    ("Q dual", rd.ObjVal),
    ("primal-dual gap", rd.ObjVal - rp.ObjVal),
], columns=["item", "value"])

## 8. 电热补救例子：系数和对偶变量

设第一阶段提前安排 $x_E$ 单位电和 $x_H$ 单位热，场景需求为：

$$
h(d)=\begin{bmatrix}80\\50\end{bmatrix}
$$

第二阶段补救变量为 $y=(y_G,y_B,y_C)^\top$，分别表示燃气机发电、锅炉供热和 CHP 联产。单位成本为：

$$
q=\begin{bmatrix}50\\40\\70\end{bmatrix}
$$

约束矩阵为：

$$
W=\begin{bmatrix}1&0&1\\0&1&1\end{bmatrix},\quad T=\begin{bmatrix}1&0\\0&1\end{bmatrix}
$$

所以两条约束分别是 $y_G+y_C\ge80-x_E$ 和 $y_B+y_C\ge50-x_H$。这里 $W$ 表示每个补救变量对电、热缺口的贡献，$T$ 表示第一阶段供应如何抵消场景需求。

若当前 $x^k=(30,10)^\top$，则剩余需求为 $(50,40)^\top$。给两条缺口约束配对偶变量 $\pi_E,\pi_H\ge0$，对偶问题为：

$$
\max_{\pi_E,\pi_H} 50\pi_E+40\pi_H
$$

$$
\text{s.t.}\quad \pi_E\le50,\quad \pi_H\le40,\quad \pi_E+\pi_H\le70
$$

$\pi_E$ 和 $\pi_H$ 分别是电力缺口、热力缺口的影子价格；Benders 割对 $x$ 的系数是 $-T^\top\pi^\star$。

In [ ]:
q_eh = np.array([50.0, 40.0, 70.0])
W_eh = np.array([[1.0, 0.0, 1.0], [0.0, 1.0, 1.0]])
h_eh = np.array([80.0, 50.0])
T_eh = np.eye(2)
xk_eh = np.array([30.0, 10.0])
b_eh = h_eh - T_eh @ xk_eh

ehp = gp.Model("electric_heat_primal", env=env)
yG = ehp.addVar(lb=0.0, name="y_G")  # 燃气机发电
yB = ehp.addVar(lb=0.0, name="y_B")  # 锅炉供热
yC = ehp.addVar(lb=0.0, name="y_C")  # CHP联产
ehp.addConstr(yG + yC >= b_eh[0], name="electric_need")  # 电力缺口
ehp.addConstr(yB + yC >= b_eh[1], name="heat_need")  # 热力缺口
ehp.setObjective(q_eh[0] * yG + q_eh[1] * yB + q_eh[2] * yC, GRB.MINIMIZE)  # 最小补救成本
ehp.optimize()

ehd = gp.Model("electric_heat_dual", env=env)
piE = ehd.addVar(lb=0.0, name="pi_E")  # 电力缺口影子价格
piH = ehd.addVar(lb=0.0, name="pi_H")  # 热力缺口影子价格
ehd.addConstr(piE <= q_eh[0], name="gas_cost_cover")  # 燃气机成本覆盖
ehd.addConstr(piH <= q_eh[1], name="boiler_cost_cover")  # 锅炉成本覆盖
ehd.addConstr(piE + piH <= q_eh[2], name="chp_cost_cover")  # CHP成本覆盖
ehd.setObjective(piE * b_eh[0] + piH * b_eh[1], GRB.MAXIMIZE)  # 最大化对偶目标
ehd.optimize()

pi_eh = np.array([piE.X, piH.X])
cut_const = pi_eh @ h_eh
cut_coef = -T_eh.T @ pi_eh

pd.DataFrame([
    ("q", q_eh),
    ("W electric row", W_eh[0]),
    ("W heat row", W_eh[1]),
    ("remaining demand h-Tx", b_eh),
    ("y_G", yG.X),
    ("y_B", yB.X),
    ("y_C", yC.X),
    ("Q primal", ehp.ObjVal),
    ("pi_E", piE.X),
    ("pi_H", piH.X),
    ("Q dual", ehd.ObjVal),
    ("primal-dual gap", ehd.ObjVal - ehp.ObjVal),
    ("Benders cut", f"theta >= {cut_const:.0f} {cut_coef[0]:+.0f} x_E {cut_coef[1]:+.0f} x_H"),
], columns=["item", "value"])

## 9. 对所有需求 $d$ 看 $Q(x,d)$

固定 $x=3$，原问题和对偶问题都给出：

$$
Q(3,d)=5\max(d-3,0)
$$

如果 $d>3$，对偶变量取 $\pi=5$；如果 $d<3$，对偶变量取 $\pi=0$。这正好是影子价格的含义：只有需求缺口真正卡住运行时，缺口约束才有正价格。

In [ ]:
ds = np.linspace(0, 10, 101)
q_values = 5 * np.maximum(ds - x_fixed, 0)
eta_current = 20.0
violations = q_values - eta_current
worst_idx = int(np.argmax(q_values))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ds, q_values, label=r"$Q(3,d)=5\max(d-3,0)$")
ax.axhline(eta_current, color="tab:orange", linestyle="--", label=r"current $\eta=20$")
ax.scatter([ds[worst_idx]], [q_values[worst_idx]], color="tab:red", zorder=3, label="worst demand")
ax.set_xlabel("demand d")
ax.set_ylabel("recourse cost")
ax.set_title("Worst scenario search for fixed x")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

pd.DataFrame([
    ("x_fixed", x_fixed),
    ("eta_current", eta_current),
    ("worst_d", ds[worst_idx]),
    ("worst_Q", q_values[worst_idx]),
    ("violation", violations[worst_idx]),
], columns=["item", "value"])

## 10. CCG max-min oracle 到单层最大化

CCG 的 oracle 是：

$$
\max_{d\in\Xi}Q(x^k,d)
$$

如果：

$$
Q(x^k,d)=\min_y\{q^\top y\mid Wy\ge h(d)-Tx^k,\ y\ge0\}
$$

则 oracle 是：

$$
\max_{d\in\Xi}\min_y\{q^\top y\mid Wy\ge h(d)-Tx^k,\ y\ge0\}
$$

对内层最小化做对偶：

$$
\min_y(\cdots)=\max_\pi\{\pi^\top(h(d)-Tx^k)\mid W^\top\pi\le q,\ \pi\ge0\}
$$

于是：

$$
\max_{d\in\Xi}\min_y(\cdots)
=
\max_{d\in\Xi}\max_\pi(\cdots)
$$

合并后：

$$
\max_{d,\pi}\quad \pi^\top(h(d)-Tx^k)
$$

$$
\text{s.t.}\quad d\in\Xi,
\quad W^\top\pi\le q,
\quad \pi\ge0
$$

这就是对偶在 CCG max-min oracle 中的作用：把嵌套的 $\max_d\min_y$ 改写成单层的 $\max_{d,\pi}$。